# Week 5 - Categories, tables and grouping

**Social and Cultural Analytics (7AAVBCS5)**

Use the same London data and methods as the lecture. Look back at the live coding whenever you need a pattern.
Copying a pattern and changing the column or argument is fine. Short written answers mean **one sentence**.

Work through sections 1–3 in order. The final section is extra practice if you have time.
You do not need to write functions, draw charts, repair identifiers or produce a long report.

Copy the download's data and weeks folders into the module folder, merging their contents. In Colab, upload this notebook and bike_trips_16-31Jul2025_with_borough.csv before running the supplied setup cell.

## Setup

Run these supplied cells. One venue-file row is a category record. One row in trips is a recorded hire; trips is loaded by the supplied setup.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a module data file."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        for local in (candidate / "data" / name, candidate / name, candidate / Path(name).name):
            if local.exists():
                return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    return downloaded / name


# Supplied load for the bike-hire overview; reused in Part III.
trips = pd.read_csv(data_path("tfl-bike-hire/bike_trips_16-31Jul2025_with_borough.csv"))

In [ ]:
venues = pd.read_csv(
    data_path("london_cultural_venues_2023.csv"), dtype={"uprn": "string"}
)
borough_context = pd.read_csv(
    data_path("london_cultural_infrastructure_2023_borough_context.csv")
)
venues[["name", "venue_type", "borough_name", "uprn"]].head()

## 1. Categories, entries and places

Start with venue type, then check what a property-identifier count would leave out.

**Task 1.1.** Convert `venue_type` to `category`, then display `.cat.categories`.

Hint: use `.astype("category")`, as in the live coding.

In [ ]:
# Your code here.

Run the filter below. It keeps only Havering's entries and preserves the category metadata.

In [ ]:
havering = venues.loc[venues["borough_name"] == "Havering"].copy()

**Task 1.2.** For Havering, print the number of possible labels and the number of observed types.

Hint: use `len(... .cat.categories)` and `.nunique()` on the venue-type column.

In [ ]:
# Your code here.

**Task 1.3.** Remove Havering's unused category labels, then print the number of possible labels again.

Hint: assign `.cat.remove_unused_categories()` back to the same column.

In [ ]:
# Your code here.

**Task 1.4.** Using the full `venues` table, count missing `uprn` values and calculate the percentage missing.

Week 4 reminder: `.isna().sum()` counts gaps; `.isna().mean() * 100` gives their percentage.

In [ ]:
# Your code here.

**Task 1.5.** Print the number of records, the number of distinct non-missing `uprn` values, and the ten most frequent `uprn` values.

Use `len(venues)`, `.nunique()` and `.value_counts().head(10)`. Look for `0`, `No UPRN` and scientific notation such as `1e+11`.

In [ ]:
# Your code here.

Run the checked example from the lecture. The same named venue appears under several labels.

In [ ]:
club = venues.loc[venues["name"] == "100 Club"]
club[["name", "venue_type", "uprn"]]

**Task 1.6.** Why is the distinct recorded UPRN count not a reliable count of operating venues? Name one problem visible above.

*Write one sentence here.*

### An ordered category: borough rates

Switch to `borough_context`: one record is one borough area. The supplied
`venues_per_10k_residents` column is a rate of venue category records, not a count of distinct places.
It combines the 2023 venue release with 2015 resident population.

Use the lecture's teaching bands: **Low** below 5, **Medium** from 5 to below 15,
and **High** from 15 upwards. These are our chosen boundaries, not official standards.

In [ ]:
borough_context[["borough_name", "venues_per_10k_residents"]].head()

**Task 1.7.** Create rate_band using the three conditions above, convert it to category, then set the order Low, Medium, High.

Copy the lecture example in small steps: create the column, assign labels with .loc, convert the dtype, then use .cat.set_categories(["Low", "Medium", "High"], ordered=True). & means both conditions must hold. These labels are ordinal because their order represents increasing rates.

In [ ]:
# Your code here.

### Switch observation units: individual hires

London's Santander Cycles are shared bikes collected from and returned to docking stations.
`trips` contains **443,539 recorded hires starting on 16–31 July 2025**; it is not a full year.
One record is one hire. The borough labels have already been supplied; no join is needed.

| Columns we use | Meaning | Data type on loading |
|---|---|---|
| `weekday` | Day on which the hire started | Text |
| `start_borough`, `end_borough` | Boroughs of the starting and ending docking stations | Text |
| `duration_min` | Recorded hire duration in minutes | Number |

**Are recorded hires longer on weekdays or at weekends?** First create the grouping category;
we will compare durations in section 3. Weekday/weekend is nominal: neither label is a higher rank.

In [ ]:
trips[["weekday", "start_borough", "end_borough", "duration_min"]].head(3)

**Task 1.8.** Create day_type as Weekday, change Saturday or Sunday hires to Weekend, and convert day_type to category.

Copy the lecture pattern. | means or; put each comparison in parentheses. This is a nominal category, not a ranking.

In [ ]:
# Your code here.

## 2. Tables

Calculate using all London entries. Select the few rows and columns below only when displaying your result.

In [ ]:
shown_boroughs = ["Camden", "Hackney", "Westminster", "Havering"]
shown_types = ["Libraries", "Music venues all", "Music nightclubs"]

**Task 2.1.** Make `type_counts` with `.value_counts()` on `venue_type`. Print its first six rows and its modal label.

Hint: `.idxmax()` gives the label attached to the largest count. This is the mode of the records, not a measure of cultural importance.

In [ ]:
# Your code here.

**Task 2.2.** Make the venue-type table as percentages and show the first six values rounded to one decimal place.

Use `.value_counts(normalize=True) * 100`, then `.round(1)`.

In [ ]:
# Your code here.

**Task 2.3.** Make the full borough-by-type crosstab as `borough_type_counts`. Display the selected boroughs and types.

Use `pd.crosstab(row_column, column_column)` on the full `venues` table. Then use `.loc[shown_boroughs, shown_types]`.

In [ ]:
# Your code here.

**Task 2.4.** Repeat the crosstab with totals. Call it `with_totals`, using `margins=True, margins_name="Total"`. Run the supplied display line after your code.

In [ ]:
# Your code here.

This display includes a Total row and column; hidden types and boroughs still contribute to the totals.

In [ ]:
print(with_totals.loc[shown_boroughs + ["Total"], shown_types + ["Total"]])

**Task 2.5.** Make `row_percent` using `normalize="index"`, multiply by 100, then display the selected boroughs and types rounded to one decimal place.

Copy Task 2.3's full-table pattern and add the normalisation argument. Do not filter the data before calculating.

In [ ]:
# Your code here.

**Task 2.6.** Copy the previous cell, change `normalize` to `"columns"`, and store the result as `column_percent`. Display the same selection.

In [ ]:
# Your code here.

**Task 2.7.** Complete this sentence from your two tables: “Music venues all entries are ___% of Camden's entries, while Camden contains ___% of London's Music venues all entries.”

*Write one sentence here.*

**Task 2.8.** Count hires by day_type, then express those counts as percentages rounded to one decimal place.

Reuse the frequency-table pattern. These counts cover 12 weekdays and four weekend days; they are not daily averages.

In [ ]:
# Your code here.

## 3. Grouping individual bike hires

Follow four decisions: grouping variable, numerical variable, calculation, interpretation. The supplied trips table contains 443,539 recorded hires starting on 16–31 July 2025. Use starting borough and recorded duration in minutes. No join is needed.

In [ ]:
trips[["start_borough", "end_borough", "duration_min"]].head()

**Task 3.1.** Group trips by start_borough as borough_groups, then count hires with .size().

Copy the two-line groupby/size pattern from the lecture.

In [ ]:
# Your code here.

**Task 3.2.** Use borough_groups to select duration_min and find the median in each starting borough.

In [ ]:
# Your code here.

**Task 3.3.** Describe duration_min, then count values equal to zero and greater than 1440 minutes. Keep the data unchanged.

In [ ]:
# Your code here.

**Task 3.4.** Make duration_summary with count, sum, mean and median. Display the results rounded to one decimal place.

Copy .agg(["count", "sum", "mean", "median"]) from the slide. count is the number of non-missing durations; sum is their total minutes; mean and median describe duration per hire in minutes. All three duration calculations retain the unusual values.

In [ ]:
# Your code here.

**Task 3.5.** Repeat the duration summary using day_type as the grouping column. Keep count, mean and median.

This is the category you created in Task 1.8. Copy the aggregation pattern and change the grouping column. As on this slide, use count, mean and median; leave out sum.

In [ ]:
# Your code here.

**Task 3.6.** Write one sentence comparing Westminster and Tower Hamlets using median duration. Include the observation and period.

*Write one sentence here.*

**Task 3.7.** Name one thing you would check before changing a zero or extremely long duration.

*Write one sentence here.*

## 4. Optional: movement between boroughs

The core worksheet is finished. Use the familiar != comparison and .loc filter, then the two-column grouping pattern from the lecture.

**Task 4.1.** Keep hires whose start_borough differs from end_borough. Call the result cross_borough.

In [ ]:
# Your code here.

**Task 4.2.** Count cross_borough hires by both start_borough and end_borough. Sort the counts largest first and show the first three.

In [ ]:
# Your code here.

**Task 4.3.** Why does a hire ending near a music venue not prove that the rider visited it?

*Write one sentence here.*